# Faz 1 — Veri Hazırlık ve Manifest

**Amaç:** TR_ABDOMEN_RAD_EMERGENCY veri setini incele, `Bilgi.xlsx`'teki ham annotasyonları doğrula, 16 ham etiketi 6 üst sınıfa eşle ve `manifest.csv` üret.

**Notebook adımları**
1. Ortam ve yol kontrolü
2. `Bilgi.xlsx` keşfi (TRAIININGDATA + COMPETITIONDATA)
3. Ham etiket → 6 üst sınıf eşleme istatistikleri
4. Tek vaka üzerinde DICOM yükleme + HU pencereleme görsel doğrulaması
5. `build_manifest()` → `outputs/splits/manifest.csv`
6. Manifest doğrulama: vaka sayısı, kesit sayısı, etiket dağılımı

**Not:** Eğitim verisi `D:/makale-pdf/Proje/abdomen/Eğitim Verisi/Eg╠åitim Verisi.zip/` altında (484 vaka klasörü). `.env` üzerinden `ABDOMEN_TRAIN_DIR` ayarlanmalıdır.

## 1. Ortam ve Yol Kontrolü

In [ ]:
!pip install python-dotenv

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# .env'i yükle (TR_ABDOMEN_BASE, TR_ABDOMEN_PROJE, TR_ABDOMEN_CODE)
load_dotenv()


In [ ]:
for k, v in os.environ.items():
    if k.startswith("ABDOMEN") or k.startswith("TR_ABDOMEN"):
        print(f"{k}={v}")

In [ ]:

# Mevcut .env değerleri config.py'nin beklediği ABDOMEN_* env'lerine map edilir
DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))

# Eğitim DICOM'ları gerçekte bu garip ad altında
EGITIM_DIR = DATA_ROOT / "Eğitim Verisi" 
YARISMA_DIR = DATA_ROOT / "Yarışma Veri Seti"   # ZIP'i ayrı klasöre açtıysanız güncelleyin

# # config.py'nin görmesi için ABDOMEN_* env'lerini set et
# os.environ["ABDOMEN_PROJECT_ROOT"] = str(PROJE)
# os.environ["ABDOMEN_DATA_ROOT"]    = str(DATA_ROOT)
# os.environ["ABDOMEN_TRAIN_DIR"]    = str(EGITIM_DIR)
# os.environ["ABDOMEN_TEST_DIR"]     = str(YARISMA_DIR)
# os.environ["ABDOMEN_BILGI_XLSX"]   = str(DATA_ROOT / "Bilgi.xlsx")
# os.environ["ABDOMEN_OUT_DIR"]      = str(CODE / "outputs")

# src/ paketini görsün
sys.path.insert(0, str(CODE))

# Doğrulama
print("DATA_ROOT   :", DATA_ROOT, "→ var?", DATA_ROOT.exists())
print("EGITIM_DIR  :", EGITIM_DIR, "→ var?", EGITIM_DIR.exists())
print("Bilgi.xlsx  :", (DATA_ROOT / "Bilgi.xlsx").exists())
print("CODE/src    :", (CODE / "src").exists())
print()
egitim_cases = sorted([d for d in EGITIM_DIR.iterdir() if d.is_dir()]) if EGITIM_DIR.exists() else []
print(f"Eğitim vaka klasörü sayısı: {len(egitim_cases)}")
print("İlk 5 vaka:", [d.name for d in egitim_cases[:5]])

## 2. Bilgi.xlsx Keşfi

In [ ]:
import pandas as pd

xl = pd.ExcelFile(DATA_ROOT / "Bilgi.xlsx")
print("Sayfalar:", xl.sheet_names)

train_df = pd.read_excel(xl, sheet_name="TRAIININGDATA")
comp_df  = pd.read_excel(xl, sheet_name="COMPETITIONDATA")

print(f"\nTRAIININGDATA   : {len(train_df):,} annotasyon, {train_df['Case Number'].nunique()} vaka")
print(f"COMPETITIONDATA : {len(comp_df):,} annotasyon, {comp_df['Case Number'].nunique()} vaka")
print(f"\nSütunlar (her ikisinde de aynı): {list(train_df.columns)}")
train_df.head(5)

In [ ]:
# Annotasyon tipi dağılımı
print("=== TRAIININGDATA — Type ===")
print(train_df['Type'].value_counts())
print("\n=== COMPETITIONDATA — Type ===")
print(comp_df['Type'].value_counts())

In [10]:
# Ham sınıf dağılımı (ilk 20)
print("=== TRAIININGDATA — Class (ilk 20) ===")
print(train_df['Class'].value_counts().head(20))

=== TRAIININGDATA — Class (ilk 20) ===
Class
Abdominal aortic aneurysm               8983
Compatible with acute pancreatitis      7128
Compatible with acute cholecystitis     5159
Gallbladder stone                       1439
Kidney stone                            1149
Abdominal aortic dissection              806
Abdominal Aorta                          733
Kidney-Bladder                           608
Pancreas                                 502
Gall bladder                             439
Colon                                    411
appendix                                 338
ureteral stone                           321
Compatible with acute appendicitis        60
Calcified diverticulum                    38
Compatible with acute diverticulitis      20
Name: count, dtype: int64


## 3. Ham Etiket → 6 Üst Sınıf Eşleme

In [11]:
from src.config import RAW_PATHOLOGY_TO_SUPER, SUPER_CLASSES, ANATOMICAL_TO_ID

print("6 ÜST SINIF:")
for i, s in enumerate(SUPER_CLASSES):
    print(f"  {i}: {s}")

print("\nHam patoloji → üst sınıf eşlemesi:")
for raw, sid in RAW_PATHOLOGY_TO_SUPER.items():
    print(f"  {raw:50s} → {sid} ({SUPER_CLASSES[sid]})")

print("\nAnatomik (Boundary Slice) sınıfları:")
for k, v in ANATOMICAL_TO_ID.items():
    print(f"  {k}: id={v}")

6 ÜST SINIF:
  0: acute_cholecystitis
  1: kidney_ureter_stone
  2: acute_pancreatitis
  3: aortic_aneurysm_dissection
  4: acute_appendicitis
  5: acute_diverticulitis

Ham patoloji → üst sınıf eşlemesi:
  Compatible with acute cholecystitis                → 0 (acute_cholecystitis)
  Gallbladder stone                                  → 0 (acute_cholecystitis)
  Kidney stone                                       → 1 (kidney_ureter_stone)
  ureteral stone                                     → 1 (kidney_ureter_stone)
  Compatible with acute pancreatitis                 → 2 (acute_pancreatitis)
  Abdominal aortic aneurysm                          → 3 (aortic_aneurysm_dissection)
  Abdominal aortic dissection                        → 3 (aortic_aneurysm_dissection)
  Compatible with acute appendicitis                 → 4 (acute_appendicitis)
  Compatible with acute diverticulitis               → 5 (acute_diverticulitis)
  Calcified diverticulum                             → 5 (acute_diverti

In [12]:
# Eşlemeye giren ve girmeyen ham etiketleri kontrol et
all_classes = pd.concat([train_df['Class'], comp_df['Class']]).value_counts()
mapped = {c: cnt for c, cnt in all_classes.items() if c in RAW_PATHOLOGY_TO_SUPER}
anat   = {c: cnt for c, cnt in all_classes.items() if c in ANATOMICAL_TO_ID}
unmapped = {c: cnt for c, cnt in all_classes.items()
            if c not in RAW_PATHOLOGY_TO_SUPER and c not in ANATOMICAL_TO_ID}

print(f"Patoloji (üst sınıfa eşlenen): {sum(mapped.values()):,} annot, {len(mapped)} sınıf")
print(f"Anatomik (boundary):           {sum(anat.values()):,} annot, {len(anat)} sınıf")
print(f"Eşlenmeyen (atılacak):         {sum(unmapped.values()):,} annot")
print("\nEşlenmeyen örnekler (eğer varsa):")
for c, n in list(unmapped.items())[:10]:
    print(f"  {c}: {n}")

Patoloji (üst sınıfa eşlenen): 35,155 annot, 10 sınıf
Anatomik (boundary):           7,293 annot, 6 sınıf
Eşlenmeyen (atılacak):         0 annot

Eşlenmeyen örnekler (eğer varsa):


## 4. Tek Vaka Görsel Doğrulama (HU Pencereleme)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from src.dicom_utils import read_dicom, dicom_to_hu, hu_to_three_channel
from src.config import DEFAULT_WINDOWS

# Annotasyonlu kesit varsa onu seç; yoksa rastgele
sample_row = train_df[train_df['Type'] == 'Bounding Box'].iloc[0]
case = int(sample_row['Case Number'])
img_id = int(sample_row['Image Id'])
dcm_path = EGITIM_DIR / str(case) / f"{img_id}.dcm"

print(f"Vaka: {case}, kesit: {img_id}")
print(f"Sınıf: {sample_row['Class']}, BBox: {sample_row['Data']}")
print(f"DICOM yolu: {dcm_path} (var: {dcm_path.exists()})")

ds = read_dicom(dcm_path)
hu = dicom_to_hu(ds)
rgb = hu_to_three_channel(hu, DEFAULT_WINDOWS)
print(f"HU shape: {hu.shape}, min={hu.min():.0f} max={hu.max():.0f}")
print(f"3 kanallı çıktı: {rgb.shape}, kanallar: {[w.name for w in DEFAULT_WINDOWS]}")

In [ ]:
# BBox parse
bbox_str = str(sample_row['Data'])
x1y1, x2y2 = bbox_str.split('-')
x1, y1 = [int(v) for v in x1y1.split(',')]
x2, y2 = [int(v) for v in x2y2.split(',')]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(hu, cmap='gray', vmin=-200, vmax=400)
axes[0].set_title(f"HU (clip [-200,400])\nVaka {case}, kesit {img_id}")
for i, w in enumerate(DEFAULT_WINDOWS):
    axes[i+1].imshow(rgb[..., i], cmap='gray')
    axes[i+1].set_title(f"{w.name}\nWL={w.level}, WW={w.width}")
for ax in axes:
    rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)
    ax.axis('off')
plt.suptitle(f"BBox: {sample_row['Class']}", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Manifest Üretimi

In [ ]:
# preprocessing.build_manifest(): tüm kesitleri (Case, Image Id) için
# super_labels, bboxes ve anatomical_boundary alanları ile CSV'ye yazar.
# CSV outputs/splits/manifest.csv'ye düşer.

from src.preprocessing import build_manifest

manifest_path = build_manifest()
print("Manifest yazıldı:", manifest_path)

In [ ]:
# Manifest doğrulama
manifest = pd.read_csv(manifest_path)
print(f"Toplam kayıt (kesit): {len(manifest):,}")
print(f"Tekil vaka: {manifest['case'].nunique()}")
print(f"Kaynak dağılımı: \n{manifest['source'].value_counts()}")
print(f"\nBBox olan kesit: {manifest['bboxes'].notna().sum() & (manifest['bboxes'] != ''):,}")

# Patoloji etiketi olan kesit sayısı
manifest['has_path'] = manifest['super_labels'].fillna('').astype(str).str.len() > 0
manifest['has_bbox'] = manifest['bboxes'].fillna('').astype(str).str.len() > 0
print(f"En az 1 üst-sınıf etiketli: {manifest['has_path'].sum():,}")
print(f"En az 1 BBox'lu kesit:       {manifest['has_bbox'].sum():,}")
manifest.head(5)

In [ ]:
# Üst sınıf dağılımı (kesit bazında — multi-label sayım)
from collections import Counter
counter = Counter()
for v in manifest['super_labels'].fillna('').astype(str):
    if not v: continue
    for s in v.split(';'):
        if s.strip(): counter[int(s)] += 1

print("Üst sınıf bazında BBox kesit sayısı:")
for sid, c in sorted(counter.items()):
    print(f"  {sid} ({SUPER_CLASSES[sid]}): {c:,}")

## 6. Faz 1 Çıktı Özeti

| Çıktı | Yol | Açıklama |
|---|---|---|
| Manifest | `outputs/splits/manifest.csv` | Tüm kesitlerin (case, image_id, source, dicom_path, super_labels, bboxes, anatomical_boundary) tablosu |

**Sonraki adım:** Faz 2'ye geçin → `Faz2_Split_Onisleme_1fold.ipynb`